In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from qgravnet import QGravNetFactory
from qgravnet.selectors import BinnedSelector
from sklearn.metrics import roc_auc_score, roc_curve
from tensorflow import keras

from hls4ml_gravnet.utils.data import load_processed, shuffle_vertices
from hls4ml_gravnet.utils.evaluation import load_run, response_rmse, get_model_coord_layers
from hls4ml_gravnet.utils.files import get_project_root_dir

PROJECT_ROOT = get_project_root_dir("hls4ml-gravnet")

## For a Single Run
 - Model Evaluation
 - Per Block: 
    - Latent Space Visualization 
    - Bin Occupancy Histogramm 
    - Bin Occupancy Heatmap
    

In [ ]:
# utility and plotting functions

def coords_probe_model(model: keras.Model) -> keras.Model:
    """Construct a probe model that outputs the predicted coordinate spaces from each QGravNet block."""

    return keras.Model(
        inputs=model.inputs,
        outputs=[l.output for l in get_model_coord_layers(model)],
        name=model.name + "_coords_probe",
    )

In [ ]:
RUN = "mini_128V_L1_S3"
model_cfg, weights_path, history, datapath, n_vertices, is_shuffled = load_run(PROJECT_ROOT / "data/results/" / RUN)
if model_cfg.get("gravnet_kwargs") is not None:
    model_cfg["gravnet_cfg"] = model_cfg.pop("gravnet_kwargs")
if model_cfg.get("selector_kwargs") is not None:
    model_cfg["selector_cfg"] = model_cfg.pop("selector_kwargs")
D = load_processed(datapath)
trained_model = QGravNetFactory(**model_cfg).create_keras_model(n_vertices, 4)
trained_model.load_weights(weights_path)
# trained_model.summary()

if is_shuffled:
    D["X_hits_test"] = shuffle_vertices(D["X_hits_test"], seed=0)
D["X_hits_test"] = D["X_hits_test"][:, :n_vertices, :]
test_energy_pred, test_pid_pred = trained_model.predict(D["X_hits_test"])

probe = coords_probe_model(trained_model)
coordinate_spaces_pred = probe.predict(D["X_hits_test"])

In [ ]:
trained_model.summary()

### AUC & RMSE

In [ ]:
test_response_rmse = response_rmse(D["y_energy_test"], test_energy_pred)
test_auc = roc_auc_score(D["y_pid_test"], test_pid_pred)
fpr, tpr, thresholds = roc_curve(D["y_pid_test"], test_pid_pred)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr)
plt.xlabel("Pion False Positive Rate")
plt.ylabel("Pion True Positive Rate")
plt.xlim(0.0, 1.0)
plt.ylim(0.0, 1.0)

plt.subplot(1, 2, 2)
plt.hist(
    test_energy_pred.flatten() / D["y_energy_test"],
    bins=50,
    # histtype="stepfilled",
    alpha=0.7,
    density=True,
)
plt.axvline(1.0, color="k", linestyle="--", lw=1, alpha=0.7)
plt.xlabel("Predicted / True Energy")
plt.ylabel("Density")
plt.xlim(0.0, 4.0)

spcr = " " * 5
notes_dataset = "Trained on small Garnet dataset \n(1 file, 10k events)" if "mini" in datapath else "Trained on full Garnet dataset \n(50 files, à 10k events)"
notes_sdim = f"Learned coordinate space dimensionality: {model_cfg['n_dimensions']}"
notes = f"{notes_dataset}\n\n{n_vertices} vertices\n\n{notes_sdim}\n"
descr = (
    "QGravNet Evaluation \n\n"
    + spcr
    + f"AUC: {test_auc:.3f} \n"
    + spcr
    + f"Response RMS: {test_response_rmse:.3f}"
    # + "\n\n\nModel Config (changes from default):\n\n"
    # + "".join([f"{spcr}{k}: {v}\n" for k, v in model_cfg.items()])
    + "\n\nNotes: \n\n"
    + notes
)
plt.text(
    1.05,
    1.0,
    descr,
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment="top",
    horizontalalignment="left",
)

plt.tight_layout()

plt.savefig(f"/scratch/lasfour/hgcal-clustering/hls4ml-gravnet/data/results/{RUN}/eval_plot.png", dpi=300)
plt.savefig(f"/scratch/lasfour/hgcal-clustering/hls4ml-gravnet/data/results/{RUN}/eval_plot.pdf")
plt.show()

### Embedded Space

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown

n_blocks = len(coordinate_spaces_pred)
block_options = [(f"Block {i}", i) for i in range(n_blocks)]
global_max_samples = max(len(coordinate_spaces_pred[i]) for i in range(n_blocks))

def plot_coords(block: int, j: int):
    coords_block = coordinate_spaces_pred[block]
    j = max(0, min(j, len(coords_block) - 1))
    sample = np.asarray(coords_block[j])

    plt.figure(figsize=(12, 5))

    if sample.shape[1] in [2, 3]:
        if sample.shape[1] == 2:
            ax = plt.subplot(1, 2, 1)
            ax.scatter(sample[:, 0], sample[:, 1], s=5)
        else:
            ax = plt.subplot(1, 2, 1, projection="3d")
            ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], s=5) 
            ax.set_zlim(-1, 1) 
        ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); 
        ax.set_title(f"Predicted coords — block={block}, sample j={j}")

        bins = np.linspace(-1, 1, 50)

        plt.subplot(1, 2, 2)
    ax2 = plt.gca()
    ax2.hist(sample[:, 0], bins=bins, alpha=0.5, label="x")
    ax2.hist(sample[:, 1], bins=bins, alpha=0.5, label="y")
    if sample.shape[1] == 3:
        ax2.hist(sample[:, 2], bins=bins, alpha=0.5, label="z")
    ax2.legend()
    ax2.set_title("Distribution of Predicted Coordinates in Single Sample")

    plt.show()

interact(
    plot_coords,
    block=Dropdown(options=block_options, value=0, description="Block"),
    j=IntSlider(value=0, min=0, max=global_max_samples - 1, step=1, description="j"),
);


In [ ]:
coordinate_spaces_pred[0].shape

In [ ]:
bins = np.linspace(-1, 1, 50)
plt.figure(figsize=(12, 5))
for i in range(len(coordinate_spaces_pred)):
    sdim = coordinate_spaces_pred[i].shape[-1]
    labels = ["x", "y"] if sdim == 2 else ["x", "y", "z"] if sdim == 3 else [f"x{d}" for d in range(sdim)]

    samples = coordinate_spaces_pred[i].reshape(-1, sdim)  
    plt.subplot(1,2,i+1)
    for j in range(sdim):
        plt.hist(samples[:, j], bins=bins, alpha=0.5, label=labels[j])
    if i != 0:
        plt.yticks([]) 
    plt.legend(title=f"Block {i+1}", loc="upper left")
plt.suptitle("Distribution of Predicted Coordinates in All Samples")
plt.tight_layout()
plt.show()

### Binning

In [ ]:
assert model_cfg.get("neighbour_selector") == "binned", "Expected selector type 'binned' in model config"
selector = BinnedSelector(**model_cfg.get("selector_cfg", model_cfg.get("selector_kwargs", {})))
coordinate_spaces_binned = [selector.compute_bins(coords) for coords in coordinate_spaces_pred]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


fig = plt.figure(figsize=(15,8), constrained_layout=True)
subfigs = fig.subfigures(nrows=len(coordinate_spaces_binned), ncols=1)

for k in range(len(coordinate_spaces_binned)):
    r = coordinate_spaces_binned[k].numpy()
    n_samples = r.shape[0]

    capacities = sorted([2])  # example bin capacity for hardware

    max_occ = np.zeros(n_samples, dtype=int)
    n_nonempty = np.zeros(n_samples, dtype=int)
    overflow_bins = {C: np.zeros(n_samples, dtype=int) for C in capacities}

    for i in range(n_samples):
        _, counts = np.unique(r[i], axis=0, return_counts=True)
        max_occ[i] = counts.max()
        n_nonempty[i] = len(counts)
        for C in capacities:
            overflow_bins[C][i] = np.sum(counts > C)

    axs = subfigs[k].subplots(1, 3)
    
    labels, counts = np.unique(max_occ, return_counts=True)
    axs[0].bar(labels, counts, width=1, align='center')
    axs[0].set_title("Per-event maximum bin occupancy")
    axs[0].set_xticks(np.unique([int(t) for t in axs[0].get_xticks()]))
    axs[0].set_xlabel("max vertices in any bin (per event)")
    axs[0].set_ylabel("events")

    labels, counts = np.unique(overflow_bins[capacities[0]], return_counts=True)
    for C in reversed(capacities):
        labels, counts = np.unique(overflow_bins[C], return_counts=True)
        axs[1].bar(labels, counts, width=0.6, align='center', label=f"C={C}", alpha=.6)
    axs[1].set_title(f"Per-event number of bins exceeding capacity")
    axs[1].set_xticks(np.arange(0, max([overflow_bins[C].max() for C in capacities])+1))
    axs[1].set_xlabel(f"# bins with occupancy exceeding capacity C")
    axs[1].legend(title="Bin capacity", loc="upper right")
    axs[1].set_ylabel("events")

    labels, counts = np.unique(n_nonempty, return_counts=True)
    axs[2].bar(labels, counts, width=.6, align='center')
    axs[2].set_title("Per-event number of non-empty bins")
    axs[2].set_xlabel("# non-empty bins")
    axs[2].set_ylabel("events")

    subfigs[k].suptitle(f"Binned KNN Occupancy Statistics (Block {k+1}) - Total Bins {model_cfg['selector_cfg']['bins_per_axis']**model_cfg['n_dimensions']}", y=1.01)

fig.get_layout_engine().set(h_pad=0.3)
plt.show()

In [ ]:
# How to force the vertices to use the full latent space